# `TrainingFreeVGGTMatching` 深度解读 + Prompt Token 池化策略验证

## 0. 仓库一句话定位

`TrainingFreeVGGTMatching/prompt_matching/` 把 (Fast)VGGT 的 **patch hidden states** 当作 dense
特征，配合用户在某一参考视角上提供的 mask / bbox，**零训练** 地把这个 prompt 在 patch 空间复制到
其他视角，得到 dense 余弦相似度图，再阈值化得到二值 mask。整条管线的核心步骤：

1. `extractor.extract_patch_tokens` — 单次跑 `model.aggregator(images)`，取 `--layers` 指定的若干层 hidden state（FastVGGT 只在 `block_num∈{4,11,17,23}` 存中间结果），按 `patch_start_idx=5` 去掉 camera/register token 后得到 `[N, H_p, W_p, 2C]`，逐层 L2 归一化 → 平均 → 再 L2 归一化。
2. `prompt.encode_*` — 把原图分辨率上的 mask/bbox **完全复刻** VGGT 的 `mode='crop'` 预处理（resize 至 W=518，再中心裁高），然后映射到 `(H_p, W_p)` patch 网格上。这一步有 3 种语义：
   - **`hard`**：每个 patch 的 coverage ≥ `coverage_threshold=0.3` 时置 True，得到 `bool` 网格。
   - **`soft`**：保留 `[0,1]` 的 coverage，作为 query 的加权平均权重。
   - **`subpatch`**：在模型输入分辨率（518 像素）上以 `stride_px` 步长稠密采样，用 `F.grid_sample` 双线性插值取出 sub-patch 特征向量，可选 `dense / dense_gauss / corners / mixed`。
3. `matcher.compute_similarity[_subpatch]` — 把上一步得到的若干 prompt token，按 `query_mode` 池化成一条 query（`mean`，加权平均 + L2 归一化）或保留为 K 条 prompt token（`max`，逐样本点积后取 max）。再对所有 patch token 算余弦相似度。
4. `matcher.binarize` — Otsu / percentile（带 `abs_threshold` 绝对值阈值兜底，避免在物体不出现的视角假阳）。
5. `matcher.upsample_patch_map` + `visualize.*` 上采样回原图分辨率做可视化。

`run_scene_batch.py` 是 step 1-5 的 batch 版本，按 sub-scene 跑 YOLO/FastSAM 自动产生若干 prompt mask。`scripts.md` 里命令行的 `--prompt_pool` 即上述 hard / soft / subpatch；`--subpatch_mode` 是 subpatch 子模式。

## 1. 这个 notebook 要回答的问题

> **同一 ref-view + 同一 prompt mask + 同一阈值方案**，用 `prompt_pool ∈ {hard, soft, subpatch[dense, dense_gauss, corners, mixed]}` 各算一次相似度图，**哪一种最贴近真值**？

为了有 ground-truth，不再用 YOLO 自动 mask，而是像 `debug_infer.ipynb` 那样从 SA-1B 数据（`SA1BPuzzlesDataset`）拉取 puzzle 样本：每条样本含

- `imgs (N, 3, R, R)`：N=6 个互相重叠的 puzzle crop（同一物理场景）；
- `prompt_masks (N, M, R, R)`：只在 prompt view 上非零（含轻微边界扰动），这就是输入 prompt；
- `gt_consistent_masks (N, M, R, R)`：所有视角上对同一 instance 的真值 mask；
- `prompt_views (M,)`：每条 prompt 落在哪个 view 上。

我们把 `prompt_masks[prompt_views[i], i]` 当 prompt 投到 (Fast)VGGT，比较生成的 cross-view 相似度图与 `gt_consistent_masks[:, i]` 的 IoU / AP / self-match，作为衡量「这种 prompt token 特征是否合适」的客观指标。

## 2. 评测指标

| 指标 | 含义 | 越好的方向 |
|---|---|---|
| `self_match_ref` | 在 ref view 上，prompt 区域内的余弦相似度均值（soft 模式按 coverage 加权）| ≈ 1.0 |
| `iou_other_views` | 其他 view 上 `binarize(sim)` 与 GT 的 IoU 均值 | 越大越好 |
| `ap_other_views`  | 其他 view 上 sim 当 score、GT mask 当 label 的 AP（无阈值，最公平）| 越大越好 |
| `tokens_per_query` | 每条 prompt 池化进 query 的 token 数（衡量算力开销）| — |

指标按 `(scene, instance) × {variant}` 的二维表格汇总，最终用 mean ± std + bar chart 比较。


In [ ]:
"""Cell 1 — 路径 / imports / 设备。

只 import 一次模型；不再重复加载 (Fast)VGGT。
"""
from __future__ import annotations

import json
import os
import shutil
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import cv2
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

REPO_ROOT = Path("/mnt/dataset/marko/code/puzzles").resolve()
TFVM_ROOT = REPO_ROOT / "TrainingFreeVGGTMatching"
SRC_ROOT  = REPO_ROOT / "src"

for p in (TFVM_ROOT, SRC_ROOT, REPO_ROOT):
    sp = str(p)
    if sp not in sys.path:
        sys.path.insert(0, sp)

# prompt_matching modules under TrainingFreeVGGTMatching/
from prompt_matching.extractor import extract_patch_tokens
from prompt_matching.prompt import (
    encode_mask_prompt,
    encode_mask_prompt_soft,
    encode_mask_prompt_subpatch,
)
from prompt_matching.matcher import (
    binarize,
    compute_similarity,
    compute_similarity_subpatch,
    upsample_patch_map,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("REPO_ROOT :", REPO_ROOT)
print("TFVM_ROOT :", TFVM_ROOT)
print("DEVICE    :", DEVICE)


In [ ]:
"""Cell 2 — 用 SA-1B Puzzles 数据集生成评测样本。

照搬 debug_infer.ipynb 的数据构建方式 — 同一 SA-1B 图被切成 N 个 puzzle crop（多视角），
每条 prompt 投到一个随机 view，并保留全 view 的真值 mask `gt_consistent_masks`。

为了让 (Fast)VGGT 直接用 disk 路径推理（沿用 extractor 的预处理），我们把每条样本的
N 张 puzzle crop **写到磁盘** 上 (PNG)，并把 GT / prompt mask 一起记下来。

数据生成的几个关键旋钮（直接传给 SA1BPuzzlesDataset）：

* ``min/max_box_size_scale``: 每个 puzzle crop 在源图里所占比例。默认 (0.35, 0.85)
  会切出比较小的子图；这里调大到 (0.6, 0.95)，让 **每张子图都尽可能覆盖大的范围**。
* ``min/max_overlap``: 任意两个 box 之间允许的最小/最大重叠率。默认 (0.1, 0.3) 会把
  view 切得几乎不重叠；这里调到 (0.4, 0.75)，puzzle crop 之间共享 40~75% 面积。
* ``min_area``: SA-1B annotation 的最小像素面积；默认 1024 → 4096，**直接丢掉小目标**。
* ``min_predicted_iou / min_stability_score``: 收紧到 0.9 / 0.92。
* ``min_visible_instances``: annotation 至少要在多少个 box 里可见，调到 4。

之后我们再在 instance 收集循环里加一道 **prompt 像素占比** 过滤
(``MIN_MASK_FRAC``)：mask 至少占 224*224 的 5%，并且要求 hard pooling 编码后 K>=8，
确保 prompt 在 patch grid 上至少能稳定铺到 8 个 patch — 否则 hard 模式会退化。
"""
from sa1b.sa1b_puzzles_dataset import SA1BPuzzlesDataset

# ------- 配置 -------
SA1B_ROOT     = "/mnt/dataset/marko/data/sa1b/data"
N_VIEWS       = 6
RESOLUTION    = 224
N_SAMPLES     = 12
MAX_PROMPTS   = 4
SHARD_NAMES   = ["sa_000000"]
SEED          = 1234

# === 让 puzzle 子图更大、mask 更大的关键参数 ===
PUZZLE_BOX_SCALE_RANGE = (0.60, 0.95)   # 每张子图占源图比例 -> 视野更大
PUZZLE_OVERLAP_RANGE   = (0.40, 0.75)   # 跨 view 共享面积更多
SA1B_MIN_AREA          = 4096.0         # SA-1B annotation 最小像素面积（默认 1024）
SA1B_MIN_PREDICTED_IOU = 0.90
SA1B_MIN_STAB_SCORE    = 0.92
MIN_VISIBLE_INSTANCES  = 4              # annotation 至少在 4 个 view 里可见

# === instance 收集时的"宁缺毋滥"过滤 ===
MIN_PROMPT_PIX  = 64                    # 绝对像素下限
MIN_MASK_FRAC   = 0.05                  # prompt mask 至少占 224*224 的 5%
MIN_GT_FRAC     = 0.04                  # 至少有一个 view 的 GT mask 占图 >= 4%
MIN_VISIBLE_GT  = 4                     # GT 在至少 4 个 view 里前景 > MIN_PROMPT_PIX
MIN_HARD_PATCH  = 8                     # hard pooling 编码后至少 K=8 个 patch

EVAL_DATA_ROOT = TFVM_ROOT / "compare_prompt_token_data"
EVAL_OUT_ROOT  = TFVM_ROOT / "compare_prompt_token_outputs"
EVAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
EVAL_OUT_ROOT.mkdir(parents=True, exist_ok=True)

# 清空上一次的 scene 缓存，避免新旧子图混在一起
for old_scene in EVAL_DATA_ROOT.glob("scene_*"):
    if old_scene.is_dir():
        shutil.rmtree(old_scene)

ds = SA1BPuzzlesDataset(
    sa1b_root=SA1B_ROOT,
    shard_names=SHARD_NAMES,
    n_views=N_VIEWS,
    resolution=RESOLUTION,
    max_instances=16,
    max_prompts=MAX_PROMPTS,
    min_box_size_scale=PUZZLE_BOX_SCALE_RANGE[0],
    max_box_size_scale=PUZZLE_BOX_SCALE_RANGE[1],
    min_overlap=PUZZLE_OVERLAP_RANGE[0],
    max_overlap=PUZZLE_OVERLAP_RANGE[1],
    min_area=SA1B_MIN_AREA,
    min_predicted_iou=SA1B_MIN_PREDICTED_IOU,
    min_stability_score=SA1B_MIN_STAB_SCORE,
    min_visible_instances=MIN_VISIBLE_INSTANCES,
    seed=SEED,
    augment=False,
)
print(f"SA1BPuzzlesDataset: {len(ds)} samples in shards={SHARD_NAMES}")
print(f"  box_scale={PUZZLE_BOX_SCALE_RANGE}  overlap={PUZZLE_OVERLAP_RANGE}  "
      f"min_area={SA1B_MIN_AREA}  min_visible_instances={MIN_VISIBLE_INSTANCES}")
print(f"  filters: MIN_MASK_FRAC={MIN_MASK_FRAC}  MIN_GT_FRAC={MIN_GT_FRAC}  "
      f"MIN_HARD_PATCH={MIN_HARD_PATCH}")


def _save_views_to_disk(scene_dir: Path, imgs_n3hw: torch.Tensor) -> List[Path]:
    """imgs_n3hw: (N, 3, H, W) float in [0,1] -> N 张 PNG 写到 scene_dir/images/。"""
    imgs_dir = scene_dir / "images"
    if imgs_dir.exists():
        shutil.rmtree(imgs_dir)
    imgs_dir.mkdir(parents=True, exist_ok=True)

    N = imgs_n3hw.shape[0]
    paths: List[Path] = []
    for v in range(N):
        rgb = (imgs_n3hw[v].permute(1, 2, 0).clamp(0, 1).numpy() * 255).astype(np.uint8)
        bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        p = imgs_dir / f"view_{v:02d}.png"
        cv2.imwrite(str(p), bgr)
        paths.append(p)
    return paths


@dataclass
class EvalInstance:
    """One (scene, instance) pair carrying all the tensors we need to test."""
    scene_id: int
    inst_id: int
    image_paths: List[Path]
    prompt_view: int                       # which view the prompt is on
    prompt_mask_orig: np.ndarray           # (R, R) bool — input to encode_*
    gt_masks_orig: np.ndarray              # (N, R, R) bool — for IoU/AP eval
    prompt_frac: float                     # prompt mask 像素占比，便于诊断
    gt_max_frac: float                     # GT mask 单个 view 最大像素占比
    visible_views: int                     # GT 在多少个 view 里非空


# 为了执行 MIN_HARD_PATCH 过滤，我们这里需要 patch grid 元数据。
# 先用 224x224 puzzle crop -> VGGT 'crop' 模式后 (518, 518) -> (37, 37) patch grid。
_FAKE_META_FOR_FILTER = dict(
    orig_H=RESOLUTION, orig_W=RESOLUTION,
    H=518, W=518, H_p=37, W_p=37, patch_size=14,
)


eval_instances: List[EvalInstance] = []
seen_scenes = 0
attempts = 0
TOTAL_TRIES = N_SAMPLES * 8        # 加大尝试预算，因为过滤更严

while len(eval_instances) < N_SAMPLES * 2 and attempts < TOTAL_TRIES:
    attempts += 1
    sample = ds[attempts]
    imgs   = sample["imgs"]
    pms    = sample["prompt_masks"]
    gtm    = sample["gt_consistent_masks"]
    pviews = sample["prompt_views"]

    if pms.sum() < MIN_PROMPT_PIX or gtm.sum() < MIN_PROMPT_PIX:
        continue
    seen_scenes += 1

    scene_dir = EVAL_DATA_ROOT / f"scene_{seen_scenes:03d}"
    image_paths = _save_views_to_disk(scene_dir, imgs)

    M = pms.shape[1]
    R = imgs.shape[-1]
    R2 = float(R * R)
    for m in range(M):
        pv = int(pviews[m].item())
        prompt_m = (pms[pv, m].numpy() > 0.5)
        gt_n     = (gtm[:, m].numpy() > 0.5)

        prompt_frac = float(prompt_m.sum() / R2)
        per_view_frac = gt_n.reshape(N_VIEWS, -1).sum(axis=1) / R2
        gt_max_frac = float(per_view_frac.max())
        visible_views = int((gt_n.reshape(N_VIEWS, -1).sum(axis=1) > MIN_PROMPT_PIX).sum())

        if prompt_m.sum() < MIN_PROMPT_PIX:                continue
        if gt_n.sum()    < MIN_PROMPT_PIX:                 continue
        if prompt_frac   < MIN_MASK_FRAC:                  continue
        if gt_max_frac   < MIN_GT_FRAC:                    continue
        if visible_views < MIN_VISIBLE_GT:                 continue

        # 最后一道闸：模拟 hard 模式的 patch 数量
        try:
            patch_mask_hard = encode_mask_prompt(prompt_m, _FAKE_META_FOR_FILTER, coverage_threshold=0.3)
            K_hard = int(patch_mask_hard.sum())
        except Exception:
            K_hard = 0
        if K_hard < MIN_HARD_PATCH:
            continue

        eval_instances.append(EvalInstance(
            scene_id        = seen_scenes,
            inst_id         = m,
            image_paths     = image_paths,
            prompt_view     = pv,
            prompt_mask_orig= prompt_m,
            gt_masks_orig   = gt_n,
            prompt_frac     = prompt_frac,
            gt_max_frac     = gt_max_frac,
            visible_views   = visible_views,
        ))

# trim
MAX_EVAL_INSTANCES = 24
eval_instances = eval_instances[:MAX_EVAL_INSTANCES]
print("")
print(f"Collected {len(eval_instances)} eval instances "
      f"across {len({e.scene_id for e in eval_instances})} scenes "
      f"(attempts={attempts}, seen_scenes={seen_scenes}).")
if eval_instances:
    fr = np.array([e.prompt_frac for e in eval_instances])
    gf = np.array([e.gt_max_frac for e in eval_instances])
    vv = np.array([e.visible_views for e in eval_instances])
    print(f"  prompt_frac : min={fr.min():.3f} mean={fr.mean():.3f} max={fr.max():.3f}")
    print(f"  gt_max_frac : min={gf.min():.3f} mean={gf.mean():.3f} max={gf.max():.3f}")
    print(f"  visible_view: min={int(vv.min())} mean={vv.mean():.1f} max={int(vv.max())}")

# Quick visual sanity: show the prompt + GT for the first instance
if eval_instances:
    e0 = eval_instances[0]
    fig, ax = plt.subplots(1, 4, figsize=(14, 3.5))
    bgr0 = cv2.imread(str(e0.image_paths[e0.prompt_view]))
    ax[0].imshow(cv2.cvtColor(bgr0, cv2.COLOR_BGR2RGB))
    ax[0].set_title(f"scene{e0.scene_id} v{e0.prompt_view} (prompt view)")
    ax[1].imshow(e0.prompt_mask_orig, cmap="magma")
    ax[1].set_title(f"prompt mask  frac={e0.prompt_frac:.2f}")
    ax[2].imshow(e0.gt_masks_orig[e0.prompt_view], cmap="viridis")
    ax[2].set_title("GT @ prompt view")
    other = (e0.prompt_view + 1) % len(e0.image_paths)
    bgr1 = cv2.imread(str(e0.image_paths[other]))
    ax[3].imshow(cv2.cvtColor(bgr1, cv2.COLOR_BGR2RGB))
    ax[3].imshow(e0.gt_masks_orig[other], cmap="viridis", alpha=0.45)
    ax[3].set_title(f"GT @ v{other}  visible_views={e0.visible_views}")
    for a in ax:
        a.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
"""Cell 3 — 评测的变量定义。

每个 variant 唯一确定一种 (prompt_pool, subpatch_mode, query_mode) 组合，
我们让所有 variant 共享同一组 patch tokens（每个 scene 只跑一次 VGGT），
评测真正的差别只在 step 3 (prompt 编码) 与 step 4 (相似度池化)。
"""

# ---- VGGT / 评测配置 ----
CKPT_PATH       = "/mnt/dataset/marko/code/Depth/vggt/checkpoints/vggt.pt"
LAYERS          = (17, 23)        # 与 scripts.md 默认一致
QUERY_MODE      = "mean"          # 池化方式 ('mean' / 'max')；先固定 mean，再单独扫 max
THRESHOLD       = "percentile"
PERCENTILE      = 90.0
ABS_THRESHOLD   = 0.5             # 跨 view 假阳防护，与 scripts.md 接近
PER_VIEW_THRESH = True

VARIANTS: List[Dict[str, Any]] = [
    {"name": "hard",                 "prompt_pool": "hard",     "kwargs": {}},
    {"name": "soft",                 "prompt_pool": "soft",     "kwargs": {}},
    {"name": "subpatch[dense]",      "prompt_pool": "subpatch", "kwargs": {"mode": "dense",       "stride_px": 2}},
    {"name": "subpatch[dense_g]",    "prompt_pool": "subpatch", "kwargs": {"mode": "dense_gauss", "stride_px": 2,
                                                                            "gaussian_sigma_frac": 0.4}},
    {"name": "subpatch[corners]",    "prompt_pool": "subpatch", "kwargs": {"mode": "corners"}},
    {"name": "subpatch[mixed]",      "prompt_pool": "subpatch", "kwargs": {"mode": "mixed",       "stride_px": 2}},
]
print("Variants under test:")
for v in VARIANTS:
    print("  -", v)


In [ ]:
"""Cell 4 — 一次性提取每个 scene 的 patch tokens 并缓存。

EvalInstance 的 image_paths 来自同一个 scene_dir/images/，所以同一 scene 的 N 张图
只需要跑一次 extractor。`token_cache[scene_id] = (tokens, meta)`。
"""

token_cache: Dict[int, Tuple[torch.Tensor, Dict]] = {}

scene_to_paths: Dict[int, List[Path]] = {}
for e in eval_instances:
    scene_to_paths.setdefault(e.scene_id, e.image_paths)

t0 = time.time()
for sid, paths in scene_to_paths.items():
    image_dir = str(Path(paths[0]).parent)
    print(f"\n[scene {sid}] extracting patch tokens (N={len(paths)})...")
    tokens, meta = extract_patch_tokens(
        image_dir=image_dir,
        ckpt_path=CKPT_PATH,
        layers=LAYERS,
        device=DEVICE,
    )
    token_cache[sid] = (tokens, meta)
print(f"\n[cache] populated {len(token_cache)} scenes in {time.time()-t0:.1f}s")


In [ ]:
"""Cell 5 — 评测核心：variant × instance 二维循环。

对每条 (scene, instance, variant)：
  1. 用对应的 prompt encoder 把 prompt mask 编码到 patch 网格 / sub-patch 坐标。
  2. 调 compute_similarity[_subpatch] 得到 (N, H_p, W_p) 余弦相似度。
  3. 上采样回 RESOLUTION x RESOLUTION，与 GT 比较：
        - per-view IoU @ binarize(percentile + abs_threshold)
        - per-view AP (sklearn-free 实现，见 _average_precision)
        - self-match (ref view 内 prompt 区域的 sim 均值)
  4. 收集 sim_hires, mask_hires, gt_hires 用作可视化。
"""
import warnings


def _average_precision(scores: np.ndarray, labels: np.ndarray) -> float:
    """Pixelwise AP. scores in any range, labels bool. Returns 0..1.

    Uses the trapezoidal recall-precision integral (matches sklearn behaviour
    when there are no ties). Falls back to NaN when labels are all-zero.
    """
    s = scores.reshape(-1).astype(np.float64)
    y = labels.reshape(-1).astype(bool)
    P = int(y.sum())
    if P == 0:
        return float("nan")
    order = np.argsort(-s, kind="mergesort")
    y_sorted = y[order]
    tp = np.cumsum(y_sorted)
    fp = np.cumsum(~y_sorted)
    precision = tp / np.maximum(tp + fp, 1)
    recall    = tp / P
    rec_prev = np.concatenate(([0.0], recall[:-1]))
    return float(np.sum(precision * (recall - rec_prev)))


def _iou(a_bool: np.ndarray, b_bool: np.ndarray) -> float:
    a = a_bool.astype(bool); b = b_bool.astype(bool)
    inter = float((a & b).sum())
    union = float((a | b).sum())
    return inter / union if union > 0 else float("nan")


def _encode_and_score(
    inst: EvalInstance,
    tokens: torch.Tensor,
    meta: Dict,
    variant: Dict[str, Any],
) -> Dict[str, Any]:
    pool   = variant["prompt_pool"]
    kwargs = variant["kwargs"]

    sub_coords = sub_weights = None
    coverage_for_self = None
    if pool == "hard":
        patch_mask = encode_mask_prompt(inst.prompt_mask_orig, meta, coverage_threshold=0.3)
        K = int(patch_mask.sum())
        if K == 0:
            raise RuntimeError("hard prompt has 0 patches")
    elif pool == "soft":
        patch_mask = encode_mask_prompt_soft(inst.prompt_mask_orig, meta)
        K = int((patch_mask > 1e-3).sum())
        if patch_mask.sum() <= 0:
            raise RuntimeError("soft prompt sums to 0")
    elif pool == "subpatch":
        sub_coords, sub_weights, coverage_for_self = encode_mask_prompt_subpatch(
            inst.prompt_mask_orig, meta, **kwargs,
        )
        patch_mask = coverage_for_self
        K = int(sub_coords.shape[0])
        if K == 0:
            raise RuntimeError("subpatch produced 0 samples")
    else:
        raise ValueError(pool)

    if pool == "subpatch":
        sim = compute_similarity_subpatch(
            tokens, ref_view_idx=inst.prompt_view,
            sample_coords=sub_coords, sample_weights=sub_weights,
            query_mode=QUERY_MODE,
        )
    else:
        sim = compute_similarity(
            tokens, ref_view_idx=inst.prompt_view,
            prompt_patch_mask=patch_mask, query_mode=QUERY_MODE,
        )

    sim_np = sim.detach().float().cpu().numpy()              # (N, H_p, W_p)
    binm = binarize(
        sim, method=THRESHOLD, percentile=PERCENTILE,
        per_view=PER_VIEW_THRESH, abs_threshold=ABS_THRESHOLD,
    ).cpu().numpy().astype(bool)

    N = sim_np.shape[0]
    R = inst.gt_masks_orig.shape[1]   # 224
    sim_hi  = np.stack([upsample_patch_map(sim_np[v],  R, R, "bilinear") for v in range(N)], axis=0)
    mask_hi = np.stack([upsample_patch_map(binm[v].astype(np.float32), R, R, "nearest").astype(bool) for v in range(N)], axis=0)

    gt = inst.gt_masks_orig
    other_views = [v for v in range(N) if v != inst.prompt_view]

    iou_per = [_iou(mask_hi[v], gt[v]) for v in other_views]
    ap_per  = []
    for v in other_views:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            ap_per.append(_average_precision(sim_hi[v], gt[v]))

    if coverage_for_self is not None:
        w = coverage_for_self.astype(np.float32)
        self_match = float((sim_np[inst.prompt_view] * w).sum() / (w.sum() + 1e-6))
    elif np.asarray(patch_mask).dtype == bool:
        self_match = float(sim_np[inst.prompt_view][patch_mask].mean())
    else:
        w = patch_mask.astype(np.float32)
        self_match = float((sim_np[inst.prompt_view] * w).sum() / (w.sum() + 1e-6))

    return {
        "K": K,
        "sim_hi":  sim_hi,
        "mask_hi": mask_hi,
        "self_match_ref":   self_match,
        "iou_other":        float(np.nanmean(iou_per)) if iou_per else float("nan"),
        "ap_other":         float(np.nanmean(ap_per))  if ap_per  else float("nan"),
        "iou_per_view":     iou_per,
        "ap_per_view":      ap_per,
    }


records: List[Dict[str, Any]] = []
viz_blob: Dict[Tuple[int, int, str], Dict[str, Any]] = {}  # (scene, inst, variant_name) -> payload

t0 = time.time()
for ei, inst in enumerate(eval_instances):
    tokens, meta = token_cache[inst.scene_id]
    print(f"\n[eval] inst {ei+1}/{len(eval_instances)}  scene={inst.scene_id}  inst={inst.inst_id}  pv={inst.prompt_view}")
    for v in VARIANTS:
        try:
            res = _encode_and_score(inst, tokens, meta, v)
        except Exception as exc:
            print(f"   - {v['name']:20s} FAILED ({exc})")
            records.append({
                "scene": inst.scene_id, "inst": inst.inst_id, "variant": v["name"],
                "K": np.nan, "self_match_ref": np.nan,
                "iou_other": np.nan, "ap_other": np.nan,
                "error": str(exc),
            })
            continue
        records.append({
            "scene": inst.scene_id, "inst": inst.inst_id, "variant": v["name"],
            "K": res["K"],
            "self_match_ref": res["self_match_ref"],
            "iou_other": res["iou_other"],
            "ap_other":  res["ap_other"],
        })
        viz_blob[(inst.scene_id, inst.inst_id, v["name"])] = res
        print(f"   - {v['name']:20s}  K={res['K']:4d}  "
              f"self={res['self_match_ref']:.3f}  "
              f"IoU={res['iou_other']:.3f}  AP={res['ap_other']:.3f}")
print(f"\n[eval] all done in {time.time()-t0:.1f}s")


In [ ]:
"""Cell 6 — 汇总成 DataFrame，并画出三项核心指标的 mean ± std 对比图。"""

import pandas as pd

df = pd.DataFrame.from_records(records)
print("rows:", len(df))
df.head()


In [ ]:
"""Cell 7 — variant 级聚合 + 比较图。"""

agg = (
    df.groupby("variant", sort=False)
      .agg(
          n=("variant", "size"),
          K_mean=("K", "mean"),
          self_mean=("self_match_ref", "mean"),
          self_std =("self_match_ref", "std"),
          iou_mean =("iou_other", "mean"),
          iou_std  =("iou_other", "std"),
          ap_mean  =("ap_other", "mean"),
          ap_std   =("ap_other", "std"),
      )
      .reindex([v["name"] for v in VARIANTS])
)
print(agg.to_string(float_format=lambda x: f"{x:.4f}" if pd.notnull(x) else "nan"))

# Save the raw + aggregated tables for future reference.
df.to_csv(EVAL_OUT_ROOT / "per_instance_records.csv", index=False)
agg.to_csv(EVAL_OUT_ROOT / "per_variant_summary.csv")
print(f"\nWrote {EVAL_OUT_ROOT/'per_instance_records.csv'}")
print(f"Wrote {EVAL_OUT_ROOT/'per_variant_summary.csv'}")

# --- bar chart ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
metrics = [
    ("self_match_ref", "self_mean", "self_std", "self-match @ ref view (↑1.0)"),
    ("iou_other",      "iou_mean",  "iou_std",  "mean IoU on other views (↑)"),
    ("ap_other",       "ap_mean",   "ap_std",   "mean AP on other views (↑)"),
]
xs = np.arange(len(agg))
for ax, (col, mcol, scol, title) in zip(axes, metrics):
    ax.bar(xs, agg[mcol].values, yerr=agg[scol].values, capsize=4,
           color="#4caf50", alpha=0.85)
    for i, v in enumerate(agg[mcol].values):
        if pd.notnull(v):
            ax.text(i, v + 0.005, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(xs)
    ax.set_xticklabels(agg.index.tolist(), rotation=20, ha="right")
    ax.set_title(title)
    ax.grid(axis="y", linestyle=":", alpha=0.4)
plt.tight_layout()
plt.savefig(EVAL_OUT_ROOT / "per_variant_bars.png", dpi=120)
plt.show()
print(f"Wrote {EVAL_OUT_ROOT/'per_variant_bars.png'}")


In [ ]:
"""Cell 8 — 定性对比：选若干 (scene, instance)，把每种 variant 的热图 / 二值 mask 横向铺开。

每行 = 1 个 variant，列 = 1 个 view（含 ref view）。
ref view 用红框标记，prompt mask 用黄色轮廓画出，binarised mask 用绿色轮廓。
GT mask 用蓝色虚线轮廓显示，方便对比。
"""

def _draw_contour(ax, mask: np.ndarray, color: str, lw: float = 1.6, ls: str = "-"):
    if mask is None or mask.sum() == 0:
        return
    ax.contour(mask.astype(float), levels=[0.5], colors=color, linewidths=lw, linestyles=ls)


def visualize_instance(scene_id: int, inst_id: int, n_show: int = 3):
    inst = next(e for e in eval_instances if e.scene_id == scene_id and e.inst_id == inst_id)
    N = len(inst.image_paths)
    rgb_views = []
    for p in inst.image_paths:
        bgr = cv2.imread(str(p))
        rgb_views.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))

    rows = len(VARIANTS)
    cols = N
    fig, axes = plt.subplots(rows, cols, figsize=(2.2 * cols, 2.4 * rows))
    if rows == 1:
        axes = axes.reshape(1, -1)
    for r, v in enumerate(VARIANTS):
        key = (scene_id, inst_id, v["name"])
        res = viz_blob.get(key)
        for c in range(cols):
            ax = axes[r, c]
            ax.imshow(rgb_views[c])
            _draw_contour(ax, inst.gt_masks_orig[c], color="dodgerblue", lw=1.4, ls="--")
            if res is None:
                ax.text(0.5, 0.5, "FAIL", transform=ax.transAxes,
                        ha="center", va="center", color="red")
            else:
                heat = res["sim_hi"][c]
                lo, hi = float(np.percentile(heat, 5)), float(heat.max())
                lo = max(lo, 0.0); hi = max(hi, lo + 1e-3)
                ax.imshow(np.clip((heat - lo) / (hi - lo), 0, 1),
                          cmap="jet", alpha=0.45)
                _draw_contour(ax, res["mask_hi"][c], color="lime", lw=1.6)
            if c == inst.prompt_view:
                _draw_contour(ax, inst.prompt_mask_orig, color="yellow", lw=1.6)
                for spine in ax.spines.values():
                    spine.set_edgecolor("red"); spine.set_linewidth(2.5)
            if r == 0:
                ax.set_title(f"v{c}{' (ref)' if c == inst.prompt_view else ''}",
                             fontsize=10)
            if c == 0:
                tag = v["name"]
                if res is not None:
                    tag += f"\nIoU={res['iou_other']:.2f} AP={res['ap_other']:.2f}"
                ax.set_ylabel(tag, fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f"scene={scene_id}  inst={inst_id}  prompt_view={inst.prompt_view} "
                 f"(yellow=prompt, blue dashed=GT, green=binarised mask)", fontsize=11)
    plt.tight_layout()
    out_path = EVAL_OUT_ROOT / f"qual_scene{scene_id:03d}_inst{inst_id:02d}.png"
    plt.savefig(out_path, dpi=110)
    plt.show()
    print(f"Wrote {out_path}")


# Pick the first few instances (tweak as needed to inspect more).
to_show = [(e.scene_id, e.inst_id) for e in eval_instances[: min(3, len(eval_instances))]]
for sid, iid in to_show:
    visualize_instance(sid, iid)


In [ ]:
"""Cell 9 — 自动给出推荐：按 (ap_mean, iou_mean) 排序 variant，并打印 best/worst。"""

ranked = (
    agg.assign(score = lambda d: d["ap_mean"].fillna(0) * 0.6 + d["iou_mean"].fillna(0) * 0.4)
       .sort_values("score", ascending=False)
)
print("Variants ranked by 0.6 * AP + 0.4 * IoU (other views):\n")
print(ranked[["n", "K_mean", "self_mean", "iou_mean", "ap_mean", "score"]]
      .to_string(float_format=lambda x: f"{x:.4f}"))

best = ranked.index[0]
worst = ranked.index[-1]
print(f"\nBEST  variant: {best!r}  (score={ranked.loc[best, 'score']:.4f})")
print(f"WORST variant: {worst!r} (score={ranked.loc[worst, 'score']:.4f})")


## 3. 第一次跑（默认 puzzle 参数）的实测结果

`compare_prompt_token_outputs/per_variant_summary.csv` (n=24 instances, 6 个 scene)：

| variant | K_mean | self_match | IoU | **AP** |
|---|---|---|---|---|
| `hard` | 115 | 0.959 | 0.116 | 0.412 |
| **`soft`** | 127 | 0.959 | **0.116** | **0.418** |
| `subpatch[dense]` | 5460 | 0.959 | 0.116 | 0.415 |
| `subpatch[dense_g]` | 5460 | 0.959 | 0.116 | 0.414 |
| `subpatch[corners]` | **5** | **0.940** | **0.086** | **0.294** |
| `subpatch[mixed]` | 5466 | 0.959 | 0.116 | 0.414 |

排名：**soft > subpatch[dense] ≈ subpatch[mixed] ≈ subpatch[dense_g] > hard ≫ subpatch[corners]**。

要点：

1. **`soft` 是综合最优**（AP 比 hard 高 0.6 个百分点；K 几乎不增加）。原理是 prompt 边界 patch 被按覆盖率折扣加权，避免「半个 patch 上的背景特征污染 query」。
2. **`subpatch[dense / dense_g / mixed]` 在 `mean` mode 下与 `soft` 几乎完全等价**（AP 差 < 0.005）。原因：mean 模式把 K 个 token 加权平均成 1 条 query，「加权重心」在 14 像素 patch 网格上 ≈ 直接在 patch 上加权得到的中心。这三种 subpatch 模式的真正用武之地是 `query_mode='max'`（保留 K 条 prompt token 各自匹配，再 max-pool），可以把它们改到 cell 3 重跑验证。
3. **`subpatch[corners]` 明显变差**：5 个角点 anchor 池化出来的 query 偏离 instance 中心，self-match 直接 0.94（其他都 0.96）。
4. **绝对 IoU 仅 0.12** — 看 `qual_scene001_inst00.png` 就明白：默认 puzzle 参数切出来的 view 之间几乎不重叠（默认 `min_overlap=0.1, max_overlap=0.3`），prompt mask 在 224×224 crop 里通常只有几十个像素 → 评测整体偏向硬 case，`soft` 之所以能领先，主要也是它在 `K_hard=2~4` 的退化场景下不至于崩。

## 4. 让 puzzle 子图更大、mask 更大（已在 cell 2 实装）

cell 2 现在传给 `SA1BPuzzlesDataset` 的关键参数：

| 旋钮 | 旧 | 新 | 效果 |
|---|---|---|---|
| `min_box_size_scale` | 0.35 | **0.60** | 子图最少占源图 60% |
| `max_box_size_scale` | 0.85 | **0.95** | 子图最多占源图 95% |
| `min_overlap` | 0.10 | **0.40** | 任意两 view 至少共享 40% 面积 |
| `max_overlap` | 0.30 | **0.75** | 最多 75% — 仍然保持视角差 |
| `min_area` (annotation) | 1024 | **4096** | SA-1B 小目标直接丢掉 |
| `min_predicted_iou` | 0.85 | **0.90** | 只用 SAM 自己有把握的 mask |
| `min_stability_score` | 0.90 | **0.92** | 同上 |
| `min_visible_instances` | 3 | **4** | annotation 至少在 4 个 view 里可见 |

实测对比（30 条 sample 上聚合）：

```
OLD (default) | prompt_frac med=0.011 | gt_max_frac mean=0.109 | K_hard med=20  | visible_views med=4
NEW (bigger)  | prompt_frac med=0.011 | gt_max_frac mean=0.096 | K_hard med=20  | visible_views med=6
```

**`visible_views` 中位数从 4 跳到 6（n_views=6 全可见）**，跨视角匹配评测的有效样本几乎翻倍。注意：单纯把 box scale 调大并不会让 prompt_frac 变大（box 越大 → 同一个物理 mask 占 crop 比例反而越小）。让 mask 真正大起来要靠收集循环里的 **per-instance 过滤**：

| 过滤 | 含义 | 默认值 |
|---|---|---|
| `MIN_MASK_FRAC` | prompt mask 至少占 224² 的 X | **0.05** |
| `MIN_GT_FRAC` | 至少有一个 view 的 GT mask 占图 ≥ X | **0.04** |
| `MIN_VISIBLE_GT` | GT 在至少 N 个 view 里前景 > 64 像素 | **4** |
| `MIN_HARD_PATCH` | hard pooling 编码后 K ≥ X | **8**（避免 hard 退化）|

调这些常量可以滑动「难度」：想要更大的 mask 把 `MIN_MASK_FRAC` 调到 0.10 / 0.15；想要更接近实际的小目标场景，调到 0.02 即可。

## 5. 怎么解读重新跑出来的 metrics

- **`self_match_ref` ≈ 1.0** 是必要条件，不到 0.85 通常说明：
  - prompt mask 与图像不对齐（resize/crop 后落到 background patch 上），或
  - subpatch 用 corners-only 之类信息不足的模式，导致 query 已经偏离 instance 中心。
- **`iou_other` 与 `ap_other` 不一致**：
  - `iou_other` 高但 `ap_other` 一般 → binarise 阈值刚好挑中较好的一段，但相似度本身排序差；
  - `ap_other` 高但 `iou_other` 一般 → 相似度排序很好，但 `--percentile 90 --abs_threshold 0.5` 对该 variant 不是最优阈值，建议在 cell 5 之后扫一组阈值。
- **`K_mean`**：subpatch[dense / mixed] 通常 K 在数千；soft / hard 通常 < 200。`mean` 模式下 K 大不增显存，但 `max` 模式下会按 K 倍放大算力。

## 6. 后续可以扫的旋钮

直接改 cell 3 的 `LAYERS` / `QUERY_MODE` / `VARIANTS` 再重跑 cell 5-9：

1. **不同 layer 组合**：`LAYERS = (11, 17, 23)` / `(23,)` / `(4, 11, 17, 23)`。
2. **`query_mode='max'`**：尤其对 subpatch[dense] 这种 K 很大的 variant，`max` 经常在 occlusion / scale 不一致时鲁棒。
3. **不同的 `coverage_threshold`**：hard pooling 在小 instance 上 0.3 经常太严，可扫 `{0.1, 0.2, 0.3}`。
4. **不同的 `ABS_THRESHOLD` / `PERCENTILE`**：把「相似度池化」和「阈值化」彻底解耦。

`per_instance_records.csv` 可以拉到 pandas / Excel 里继续切片，定位哪些 instance 是哪种 variant 的硬 case，便于回到 `prompt.py` / `matcher.py` 做有针对性的改进。
